# 🤟 Upgraded HST-GNN v2 — Sign Language Translation

**Nâng cấp từ:** arxiv 2111.07258 (Kan et al., 2021)

| Component | Bài gốc | v2 nâng cấp |
|---|---|---|
| Feature | ResNet-152 + TVL1-flow | MediaPipe keypoints (offline, free) |
| Graph | HST-GNN (bilinear adj) | HST-GNN Lite (cosine adj, pre-LN) |
| Temporal | 2×LSTM | 1D Conv + **Sliding-Window Transformer** |
| Text decoder | 2-stage LSTM | mBART-50 pretrained |
| Loss routing | CTC + CE (tất cả) | **CTC chỉ cho Continuous / CE cho Isolated** |
| Training | Single-phase | **2-phase (encoder pretrain → decoder finetune)** |
| Checkpoint | Cuối epoch | **Auto-save Drive mỗi 15 phút** (Colab-safe) |
| Dataset | PHOENIX only | **Multi-dataset (PHOENIX + WLASL + ASL từ Kaggle)** |

## 🔑 Chiến lược train với Colab FREE:
```
Session 1 (Phase 1, ~2h): Pretrain Encoder → CTC gloss recognition
Session 2 (Phase 2, ~2h): Fine-tune mBART Decoder → text translation  
Session 3 (Phase 0, ~2h): End-to-end joint fine-tune
```
→ Mỗi session tự động save lên Drive mỗi 15 phút.
→ Session sau load checkpoint Drive → train tiếp.

## 0️⃣ Mount Google Drive (BẮT BUỘC — lưu checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/sign_language_v2'
DRIVE_CKPT = f'{DRIVE_ROOT}/checkpoints'
DRIVE_DATA = f'{DRIVE_ROOT}/data'

import os
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
print(f'✓ Drive mounted')
print(f'  Checkpoint dir: {DRIVE_CKPT}')
print(f'  Data dir:       {DRIVE_DATA}')

## 1️⃣ Check GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

# T4 = 15GB, P100 = 16GB, A100 = 40/80GB
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 14:
    print('⚠️  VRAM < 14GB — dùng batch_size=2, d_model=128')
    BATCH_SIZE = 2
    D_MODEL = 128
elif vram_gb < 20:
    print('✓ T4/P100 — dùng batch_size=4, d_model=256')
    BATCH_SIZE = 4
    D_MODEL = 256
else:
    print('🚀 A100 — dùng batch_size=8, d_model=256')
    BATCH_SIZE = 8
    D_MODEL = 256

print(f'→ batch_size={BATCH_SIZE}, d_model={D_MODEL}')

## 2️⃣ Install Dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 accelerate
!pip install kagglehub[pandas-datasets]
!pip install mediapipe opencv-python-headless
!pip install pyyaml tqdm einops
print('✓ Done')

In [ ]:
print('✓ All dependencies installed')

## 3️⃣ Upload Code Files (từ GitHub hoặc local)

In [ ]:
# Option A: Clone từ GitHub (nếu đã push code lên)
# !git clone https://github.com/YOUR_USERNAME/sign-language-hstgnn
# %cd sign-language-hstgnn

# Option B: Upload thủ công từ máy
import os
os.makedirs('/content/sign_language', exist_ok=True)

from google.colab import files
print('Upload 7 files: train.py, model.py, dataset.py, config.py, utils.py, vocabulary.py, scheduler.py')
uploaded = files.upload()

for fname in uploaded:
    dest = f'/content/sign_language/{fname}'
    os.rename(fname, dest)
    print(f'  Moved: {fname}')

%cd /content/sign_language
!ls -la

## 4️⃣ Kaggle API Setup

**Cách lấy token (1 lần duy nhất):**
1. Vào https://www.kaggle.com/settings
2. Mục **API** → **Generate New Token** → tải xuống `kaggle.json`
3. Hoặc dùng **Colab Secrets** (ổn định hơn, không cần upload lại mỗi session)

In [ ]:
# ── Cách 1: Dùng Colab Secrets (KHUYẾN NGHỊ — không cần upload lại) ──
# Vào menu: Chỉnh sửa → Bí mật (🔒) → Thêm KAGGLE_USERNAME và KAGGLE_KEY

import os

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('✓ Kaggle credentials loaded from Colab Secrets')
except Exception:
    # ── Cách 2: Upload kaggle.json ──
    print('Secrets không có → Upload kaggle.json...')
    from google.colab import files
    uploaded = files.upload()
    import shutil
    os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
    shutil.move('kaggle.json', os.path.expanduser('~/.config/kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.config/kaggle/kaggle.json'), 0o600)
    print('✓ Kaggle API configured from kaggle.json')

# Test
import kagglehub
print('✓ kagglehub imported successfully')

## 5️⃣ Download Datasets từ Kaggle

Chọn dataset phù hợp với mục tiêu:

| Dataset | Loại | Ngôn ngữ | Kích thước | Dùng cho |
|---------|------|----------|------------|----------|
| `risangbaskoro/wlasl-processed` | Isolated | ASL | ~7GB | Nhận diện ký hiệu |
| `grassknoted/asl-alphabet` | Isolated | ASL | ~1GB | Pre-training nhẹ |
| `datamunge/sign-language-mnist` | Isolated | ASL | ~1MB | Quick test |
| PHOENIX-2014T | Continuous | DGS | ~1.7GB | Translation |
| `sttaseen/wlasl2000-resized` | Isolated | ASL | ~4GB | Balanced training |

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os, json, shutil
import numpy as np
from pathlib import Path

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/data/keypoints', exist_ok=True)

# ────────────────────────────────────────
# Dataset 1: Sign Language MNIST (NHANH — dùng để test pipeline trước)
# Đây là isolated signs (A-Z), mode="isolated" → chỉ CE loss
# ────────────────────────────────────────
print('📦 Downloading Sign Language MNIST (isolated, quick test)...')
df_mnist_train = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'datamunge/sign-language-mnist',
    'sign_mnist_train/sign_mnist_train.csv',
)
df_mnist_test = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'datamunge/sign-language-mnist',
    'sign_mnist_test/sign_mnist_test.csv',
)
print(f'  Train: {df_mnist_train.shape}, Test: {df_mnist_test.shape}')

In [ ]:
# ────────────────────────────────────────
# Dataset 2: WLASL-processed (Isolated ASL, ~2000 signs từ video)
# Download file annotations (JSON)
# ────────────────────────────────────────
print('📦 Downloading WLASL metadata...')
try:
    wlasl_path = kagglehub.dataset_download(
        'risangbaskoro/wlasl-processed',
        output_dir='/content/data/wlasl_raw'
    )
    print(f'  WLASL downloaded to: {wlasl_path}')
    # List files
    import glob
    files_list = glob.glob('/content/data/wlasl_raw/**/*', recursive=True)[:20]
    for f in files_list:
        print(f'  {f}')
except Exception as e:
    print(f'  WLASL download failed: {e}')
    print('  → Sẽ dùng MNIST làm dataset chính')

## 6️⃣ Prepare Data — Convert sang format chuẩn

In [ ]:
import sys
sys.path.insert(0, '/content/sign_language')

import json, random
import numpy as np
from pathlib import Path
from vocabulary import Vocabulary

Path('/content/data').mkdir(exist_ok=True)

# ══════════════════════════════════════════
# 6a. Convert Sign Language MNIST → JSON
#     dataset_mode = "isolated" (auto-detected)
# ══════════════════════════════════════════

LABEL_MAP = {i: chr(ord('A') + i) for i in range(26)}
# MNIST không có J(9) và Z(25) vì cần chuyển động

def mnist_row_to_sample(row, idx, split):
    label = int(row['label'])
    char = LABEL_MAP.get(label, str(label))
    pixel_cols = [c for c in row.index if c.startswith('pixel')]
    pixels = row[pixel_cols].values.astype(np.float32) / 255.0  # normalize
    img = pixels.reshape(28, 28)  # 28x28

    # Tạo fake keypoints từ ảnh 28x28 (placeholder cho pipeline test)
    # Thực tế: nên extract MediaPipe từ ảnh gốc
    # Nhưng MNIST chỉ có 28x28 grayscale, không đủ resolution
    fake_kps = np.zeros((5, 543, 4), dtype=np.float32)  # T=5 frames
    # Dùng 28*28=784 giá trị đầu làm proxy features
    flat = np.tile(pixels, 5).reshape(5, -1)
    n_fill = min(flat.shape[1] // 4, 543)
    fake_kps[:, :n_fill, :] = flat[:, :n_fill*4].reshape(5, n_fill, 4)

    # Lưu keypoints ra file .npy
    kp_path = f'/content/data/keypoints/mnist_{split}_{idx}.npy'
    np.save(kp_path, fake_kps)

    return {
        'id': f'mnist_{split}_{idx}',
        'keypoints_path': kp_path,
        'gloss': char,
        'text': char,                   # isolated: gloss = text
        'dataset': 'sign-language-mnist',
        'dataset_mode': 'isolated',     # explicit mode → chỉ dùng CE loss
        'split': split,
    }

print('Converting MNIST train...')
train_samples = []
val_samples = []
test_samples = []

mnist_train_list = []
for i, row in df_mnist_train.iterrows():
    mnist_train_list.append(row)

random.shuffle(mnist_train_list)
n = len(mnist_train_list)
n_train = int(n * 0.85)
n_val = int(n * 0.10)

for i, row in enumerate(mnist_train_list[:n_train]):
    train_samples.append(mnist_row_to_sample(row, i, 'train'))
for i, row in enumerate(mnist_train_list[n_train:n_train+n_val]):
    val_samples.append(mnist_row_to_sample(row, i, 'val'))
for i, row in enumerate(mnist_train_list[n_train+n_val:]):
    val_samples.append(mnist_row_to_sample(row, i, 'val_extra'))

print('Converting MNIST test...')
for i, row in df_mnist_test.iterrows():
    test_samples.append(mnist_row_to_sample(row, i, 'test'))

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

In [ ]:
# ══════════════════════════════════════════
# 6b. Merge WLASL samples nếu download thành công
#     WLASL = isolated, dataset_mode="isolated"
# ══════════════════════════════════════════

import glob, os

wlasl_json_files = glob.glob('/content/data/wlasl_raw/**/*.json', recursive=True)
print(f'Found {len(wlasl_json_files)} WLASL JSON files')

wlasl_samples = []
for jf in wlasl_json_files[:5]:  # preview
    try:
        with open(jf) as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f'  {jf}: {len(data)} samples, first keys: {list(data[0].keys()) if data else []}')
        elif isinstance(data, dict):
            print(f'  {jf}: dict with keys: {list(data.keys())[:5]}')
    except Exception as e:
        print(f'  {jf}: error {e}')

# WLASL format thường là:
# [{"gloss": "book", "instances": [{"video_id": "...", "split": "train", ...}]}]
# Nếu đã có keypoints .npy thì add thẳng vào dataset

wlasl_npy_files = glob.glob('/content/data/wlasl_raw/**/*.npy', recursive=True)
print(f'Found {len(wlasl_npy_files)} WLASL .npy keypoint files')
for f in wlasl_npy_files[:3]:
    arr = np.load(f)
    print(f'  {f}: shape={arr.shape}')

In [ ]:
# ══════════════════════════════════════════
# 6c. Lưu data JSON + Build vocabulary
# ══════════════════════════════════════════

# Merge tất cả datasets
all_train = train_samples + [s for s in wlasl_samples if s.get('split') == 'train']
all_val   = val_samples   + [s for s in wlasl_samples if s.get('split') != 'train']

with open('/content/data/train.json', 'w') as f:
    json.dump(all_train, f)
with open('/content/data/val.json', 'w') as f:
    json.dump(all_val, f)
with open('/content/data/test.json', 'w') as f:
    json.dump(test_samples, f)

print(f'Saved: train={len(all_train)}, val={len(all_val)}, test={len(test_samples)}')

# Build vocabularies
gloss_vocab = Vocabulary.build_from_json('/content/data/train.json', 'gloss')
text_vocab  = Vocabulary.build_from_json('/content/data/train.json', 'text')

gloss_vocab.save('/content/data/gloss_vocab.json')
text_vocab.save('/content/data/text_vocab.json')

print(f'Gloss vocab: {len(gloss_vocab)} tokens')
print(f'Text vocab:  {len(text_vocab)} tokens')
print(f'Sample train[0]: {json.dumps(all_train[0], default=str)[:300]}')

## 7️⃣ Configure Training

In [ ]:
import yaml, os
os.makedirs('/content/sign_language/configs', exist_ok=True)

# Config tự động điều chỉnh theo VRAM
config_colab = {
    # Data
    'train_data_path': '/content/data/train.json',
    'val_data_path':   '/content/data/val.json',
    'test_data_path':  '/content/data/test.json',
    'gloss_vocab_path': '/content/data/gloss_vocab.json',
    'text_vocab_path':  '/content/data/text_vocab.json',
    'num_keypoints': 543,
    'keypoint_dim': 4,
    'max_seq_len': 256,
    'max_text_len': 64,
    'num_workers': 2,

    # Model
    'd_model': D_MODEL,
    'num_graph_layers': 2,
    'num_temporal_layers': 3,
    'num_heads': 8,
    'temporal_window_size': 5,   # Sliding window (bài gốc: window ≤ 3-5)
    'dropout': 0.1,
    'decoder_name': 'facebook/mbart-large-50',
    'gradient_checkpointing': True,

    # Training
    'epochs': 30,
    'batch_size': BATCH_SIZE,
    'eval_batch_size': BATCH_SIZE * 2,
    'gradient_accumulation_steps': 8,
    'lr': 5e-4,
    'pretrained_lr_scale': 0.05,
    'weight_decay': 1e-4,
    'max_grad_norm': 1.0,
    'warmup_ratio': 0.1,
    'patience': 8,
    'use_amp': True,
    'seed': 42,

    # Loss weights
    'lambda_ctc': 0.5,
    'lambda_ce': 0.5,

    # Augmentation
    'aug_flip_prob': 0.5,
    'aug_noise_std': 0.01,
    'aug_scale_range': [0.85, 1.15],
    'aug_rotation': 15.0,
    'aug_time_stretch': [0.8, 1.2],
    'aug_frame_drop': 0.1,
    'aug_joint_drop': 0.05,

    # Logging
    'log_interval': 20,
    'output_dir': '/content/outputs/',
    'drive_backup': DRIVE_CKPT,
}

config_path = '/content/sign_language/configs/colab.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config_colab, f)

print('✓ Config saved:')
for k, v in config_colab.items():
    print(f'  {k}: {v}')

## 8️⃣ Sanity Check — Forward Pass

In [ ]:
import sys, torch
sys.path.insert(0, '/content/sign_language')

from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary

config = get_config('/content/sign_language/configs/colab.yaml')
print(f'Device: {config.device}')

gloss_vocab = Vocabulary.load('/content/data/gloss_vocab.json')
text_vocab  = Vocabulary.load('/content/data/text_vocab.json')

model = UpgradedHSTGNN(
    num_keypoints=config.num_keypoints,
    keypoint_dim=config.keypoint_dim,
    d_model=config.d_model,
    num_graph_layers=config.num_graph_layers,
    num_heads=config.num_heads,
    num_gloss_classes=len(gloss_vocab),
    text_vocab_size=len(text_vocab),
    temporal_window_size=getattr(config, 'temporal_window_size', 5),
    decoder_name=config.decoder_name,
    dropout=config.dropout,
    use_gradient_checkpointing=config.gradient_checkpointing,
).to(config.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total/1e6:.2f}M')
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Frozen params:    {(total-trainable)/1e6:.2f}M (mBART lower layers)')

# Forward pass test với sliding window attention
B, T, N, C = 2, 32, 543, 4
kps    = torch.randn(B, T, N, C).to(config.device)
kp_lens = torch.tensor([32, 28], device=config.device)
gloss_t = torch.randint(1, max(len(gloss_vocab), 2), (B, 5)).to(config.device)
text_t  = torch.randint(1, max(len(text_vocab), 2), (B, 10)).to(config.device)

with torch.cuda.amp.autocast():
    out = model(kps, kp_lens, gloss_t, text_t)

print('\n✓ Forward pass OK!')
print(f"  gloss_log_probs: {out['gloss_log_probs'].shape}")
print(f"  text_logits:     {out['text_logits'].shape}")
print(f"  encoder_hidden:  {out['encoder_hidden'].shape}")

# Check VRAM sau forward pass
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'\nVRAM used: {vram_used:.2f} GB')

## 9️⃣ Auto-Resume Logic

Mỗi khi session mới bắt đầu, cell này tự kiểm tra Drive xem có checkpoint cũ không.
Nếu có → tự động restore và train tiếp từ đúng epoch.

In [ ]:
import os, shutil
from pathlib import Path

LOCAL_OUTPUT = '/content/outputs'
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

local_latest = f'{LOCAL_OUTPUT}/checkpoint_latest.pt'
drive_latest = f'{DRIVE_CKPT}/checkpoint_latest.pt'

RESUME_PATH = None
TRAINING_PHASE = 0  # 0=joint, 1=encoder only, 2=decoder only

if os.path.exists(local_latest):
    RESUME_PATH = local_latest
    print(f'✓ Found local checkpoint: {local_latest}')

elif os.path.exists(drive_latest):
    print(f'Restoring checkpoint from Drive...')
    shutil.copy(drive_latest, local_latest)
    RESUME_PATH = local_latest
    print(f'✓ Restored checkpoint from Drive: {drive_latest}')

    # Kiểm tra có checkpoint_best không
    drive_best = f'{DRIVE_CKPT}/checkpoint_best.pt'
    if os.path.exists(drive_best):
        shutil.copy(drive_best, f'{LOCAL_OUTPUT}/checkpoint_best.pt')
        print(f'✓ Restored best checkpoint from Drive')

else:
    print('No checkpoint found → Starting fresh training')
    print(f'Phase: {TRAINING_PHASE} (0=joint, 1=encoder, 2=decoder)')

print(f'\nResume path: {RESUME_PATH}')
print(f'Training phase: {TRAINING_PHASE}')

# Hiển thị tất cả checkpoints trên Drive
if os.path.exists(DRIVE_CKPT):
    ckpts = sorted(os.listdir(DRIVE_CKPT))
    print(f'\nCheckpoints on Drive ({len(ckpts)} files):')
    for c in ckpts:
        size = os.path.getsize(f'{DRIVE_CKPT}/{c}') / 1e6
        print(f'  {c}: {size:.1f} MB')

## 🚀 10. Train!

### Thứ tự chạy theo session:

**Session 1 — Phase 1 (Pretrain Encoder, ~2h):**
```python
TRAINING_PHASE = 1   # Freeze mBART, chỉ train Graph+Temporal Encoder
```

**Session 2 — Phase 2 (Fine-tune Decoder, ~2h):**
```python
TRAINING_PHASE = 2   # Freeze Encoder, chỉ train mBART Decoder
```

**Session 3 — Phase 0 (Joint Fine-tune, ~2h):**
```python
TRAINING_PHASE = 0   # Train cả model với LR nhỏ
```

In [ ]:
# ── Đặt phase trước khi train ──
# Session 1: phase=1, Session 2: phase=2, Session 3: phase=0
TRAINING_PHASE = 1   # thay đổi theo session

# ── Build command ──
import sys
cmd_parts = [
    sys.executable, '/content/sign_language/train.py',
    '--config', '/content/sign_language/configs/colab.yaml',
    '--output_dir', LOCAL_OUTPUT,
    '--drive_backup', DRIVE_CKPT,
    '--phase', str(TRAINING_PHASE),
    '--save_interval_min', '15',    # Save Drive mỗi 15 phút
]

if RESUME_PATH:
    cmd_parts += ['--resume', RESUME_PATH]

cmd = ' '.join(cmd_parts)
print('Command:')
print(cmd)
print()
print('Training sẽ:')
print(f'  - Auto-save Drive mỗi 15 phút')
print(f'  - Save checkpoint_epochXXX.pt cuối mỗi epoch')
print(f'  - Nếu Colab die: chạy lại cell "Auto-Resume" ở trên để restore')

In [ ]:
# ── Chạy training ──
!{cmd}

## 📊 11. Visualize Training History

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

history_path = f'{LOCAL_OUTPUT}/history.json'
if not os.path.exists(history_path):
    print('No history.json yet — train first!')
else:
    with open(history_path) as f:
        history = json.load(f)

    epochs = [h['epoch'] for h in history]
    wer    = [h.get('wer', 100) for h in history]
    bleu4  = [h.get('bleu4', 0) for h in history]
    loss   = [h.get('loss', 0) for h in history]
    ctc_l  = [h.get('ctc_loss', 0) for h in history]
    ce_l   = [h.get('ce_loss', 0) for h in history]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('HST-GNN v2 Training Progress', fontsize=15, fontweight='bold')

    axes[0,0].plot(epochs, loss, 'b-o', ms=3); axes[0,0].set_title('Total Loss ↓')
    axes[0,1].plot(epochs, ctc_l, 'g-o', ms=3); axes[0,1].set_title('CTC Loss ↓')
    axes[0,2].plot(epochs, ce_l, 'm-o', ms=3); axes[0,2].set_title('CE Loss ↓')
    axes[1,0].plot(epochs, wer, 'r-o', ms=3); axes[1,0].set_title('Val WER ↓')
    axes[1,1].plot(epochs, bleu4, 'teal', marker='o', ms=3); axes[1,1].set_title('Val BLEU-4 ↑')

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('Epoch')

    axes[1,2].axis('off')
    axes[1,2].text(0.1, 0.7, f'Best WER: {min(wer):.2f}%\n(epoch {epochs[wer.index(min(wer))]})',
                   transform=axes[1,2].transAxes, fontsize=14, color='red')
    axes[1,2].text(0.1, 0.4, f'Best BLEU-4: {max(bleu4):.2f}\n(epoch {epochs[bleu4.index(max(bleu4))]})',
                   transform=axes[1,2].transAxes, fontsize=14, color='teal')

    plt.tight_layout()
    save_path = f'{LOCAL_OUTPUT}/training_history.png'
    plt.savefig(save_path, bbox_inches='tight')
    # Backup hình lên Drive
    import shutil
    shutil.copy(save_path, f'{DRIVE_ROOT}/training_history.png')
    plt.show()
    print(f'Saved to: {save_path}')

## 🧪 12. Evaluate Best Model

In [ ]:
best_ckpt = f'{LOCAL_OUTPUT}/checkpoint_best.pt'
if not os.path.exists(best_ckpt):
    # Restore từ Drive
    drive_best = f'{DRIVE_CKPT}/checkpoint_best.pt'
    if os.path.exists(drive_best):
        shutil.copy(drive_best, best_ckpt)
        print('✓ Restored best checkpoint from Drive')

!python /content/sign_language/train.py \
    --config /content/sign_language/configs/colab.yaml \
    --resume {best_ckpt} \
    --eval_only \
    --output_dir {LOCAL_OUTPUT}

## ❓ 13. Khi Colab hết free GPU — Làm gì tiếp?

### Option A: Kaggle Notebooks (**KHUYẾN NGHỊ nhất**)
```
✓ 30h GPU/tuần (P100 16GB) — hoàn toàn MIỄN PHÍ
✓ Upload code lên Kaggle Dataset → import vào notebook
✓ Copy checkpoint từ Google Drive xuống Kaggle
✓ Tiếp tục train với --resume
```

### Cách chuyển checkpoint Colab → Kaggle:
```python
# Trên Colab: upload checkpoint lên Kaggle Dataset
import kagglehub
kagglehub.dataset_upload(
    'your_username/sign-language-checkpoints',
    '/content/drive/MyDrive/sign_language_v2/checkpoints'
)
```

```python
# Trên Kaggle: download checkpoint
import kagglehub
path = kagglehub.dataset_download('your_username/sign-language-checkpoints')
```

### Option B: Lightning.AI (22h free GPU/tháng)
- Persistent workspace — file không mất giữa sessions
- Không cần Drive backup

### Option C: Google Colab Pro ($10/tháng)
- A100 80GB — batch_size=32, không cần gradient_accumulation
- Train xong trong 1 session

### 📋 Chiến lược chia session cho Colab FREE:
| Session | Phase | Thời gian | Targets |
|---------|-------|-----------|--------|
| 1 | Phase 1 (encoder) | ~2h | CTC WER < 50% |
| 2 | Phase 2 (decoder) | ~2h | BLEU-4 > 5 |
| 3 | Phase 0 (joint) | ~2h | WER < 30%, BLEU-4 > 10 |
| 4+ | Phase 0 (resume) | ~2h/session | Fine-tune thêm |

## 🎬 14. Inference Demo

In [ ]:
import torch, sys
sys.path.insert(0, '/content/sign_language')

from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary
import numpy as np

config = get_config('/content/sign_language/configs/colab.yaml')
gloss_vocab = Vocabulary.load('/content/data/gloss_vocab.json')
text_vocab  = Vocabulary.load('/content/data/text_vocab.json')

# Load best model
model = UpgradedHSTGNN(
    num_keypoints=config.num_keypoints,
    keypoint_dim=config.keypoint_dim,
    d_model=config.d_model,
    num_graph_layers=config.num_graph_layers,
    num_heads=config.num_heads,
    num_gloss_classes=len(gloss_vocab),
    text_vocab_size=len(text_vocab),
    temporal_window_size=getattr(config, 'temporal_window_size', 5),
    decoder_name=config.decoder_name,
).to(config.device)

ckpt_path = f'{LOCAL_OUTPUT}/checkpoint_best.pt'
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=config.device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    print('✓ Model loaded!')
else:
    print('No checkpoint found — train first!')

def translate_video(video_path: str) -> dict:
    """Dịch video ký hiệu → văn bản."""
    from dataset import extract_keypoints_from_video, KeypointNormalizer

    kps = extract_keypoints_from_video(video_path, output_path=None)
    if kps is None:
        return {'error': 'Cannot extract keypoints'}

    normalizer = KeypointNormalizer()
    kps = normalizer(kps)

    kps_t = torch.from_numpy(kps).unsqueeze(0).to(config.device)
    lengths = torch.tensor([kps.shape[0]], device=config.device)

    with torch.no_grad():
        out = model(kps_t, lengths)

    gloss_ids = model.decode_gloss(out['gloss_log_probs'])
    gloss_text = gloss_vocab.ids_to_text(gloss_ids[0])

    text_ids = model.decode_text(
        out['encoder_hidden'], out['encoder_lengths'],
        text_vocab, config.max_text_len
    )
    translation = text_vocab.ids_to_text(text_ids[0])

    return {'gloss': gloss_text, 'translation': translation}

print('✓ Inference pipeline ready!')
print('Gọi: result = translate_video("path/to/video.mp4")')
print('Hoặc upload video:')
print('  from google.colab import files')
print('  uploaded = files.upload()')
print('  result = translate_video(list(uploaded.keys())[0])')